# Solubility, Yield, and Physicochemical Properties of Escherichia coli Cytoplasmic Proteins under Macromolecular Crowding Conditions Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR^2 dataset](https://doi.org/10.71728/senscience.96z4-183q) using the `mlcroissant` library and a Croissant schema.

### Dataset Source
The [dataset's Croissant schema is available here](https://sen.science/doi/10.71728/senscience.96z4-183q/fair2.json) and includes 
* Tabular measurements for 142 _E. coli_ K-12 cytoplasmic proteins,
* Values for solubility, yield, molecular weight, isoelectric point, amino acid composition, superfamily annotation, and more,
* Distinct record sets representing different granularities of protein data, each described by unique `@id` values.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant JSON-LD)
croissant_url = 'https://sen.science/doi/10.71728/senscience.96z4-183q/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\n\nDescription: {metadata.description}\n")
print(f"License: {metadata.license}")
print(f"Version: {metadata.version}")

## 2. Data Overview

List out available record sets and their corresponding `@id` fields. Also, display the available fields (`@id`) for each record set.

**Note:** All dataset entities are referenced by their unique `@id` field. This ensures unambiguous selection of record sets and fields.

In [ ]:
# Get available record sets by @id
record_sets = list(dataset.record_sets)
print("Available record sets in the dataset (by @id):")
for i, recset in enumerate(record_sets):
    print(f"{i+1}. {recset['@id']}")
    print(f"    Name: {recset.get('name', '(no name)')}")
    print(f"    Description: {recset.get('description', '(no description)')}")

# For the first record set, list its field @ids and descriptions
if len(record_sets) > 0:
    first_recset_id = record_sets[0]['@id']
    print(f"\nFields for record set '@id': {first_recset_id}")
    fields = record_sets[0].get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        if isinstance(field, dict):
            print(f"  - {field.get('@id', 'Unknown')} : {field.get('name', '(no name)')}")
        elif isinstance(field, str):
            print(f"  - {field}")

## 3. Data Extraction

Below, we demonstrate how to extract data from each record set using their `@id`. We then load one record set as a DataFrame for exploration.

In [ ]:
# Gather all record set @ids to extract records
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set: {record_set_id}")
    else:
        print(f"No records found for record set: {record_set_id}")

# Display the columns of the first loaded record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id in dataframes:
    print(f"\nColumns in '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's demonstrate some analysis for one numeric field in the main record set using its unique `@id`.

- Filtering by a numeric value
- Normalizing data
- Grouping by a categorical field

Replace the `numeric_field_id` and `group_field_id` variables with actual `@id` values from the Data Overview.

In [ ]:
# Example field IDs (replace with actual field @id values from the overview if needed)
# For demonstration, choose the first numeric column and a categorical/grouping field.

# Select the main DataFrame
df = None
if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]

if df is not None and len(df) > 0:
    # Try to automatically find a likely numeric field (e.g., solubility, yield)
    numeric_candidates = [col for col in df.columns if df[col].dtype in [int, float] or pd.api.types.is_numeric_dtype(df[col])]
    if len(numeric_candidates) == 0:
        # Try coercing all columns to numeric (ignore errors)
        possible = []
        for col in df.columns:
            try:
                df[col+'_num'] = pd.to_numeric(df[col], errors='coerce')
                if df[col+'_num'].notna().sum() > 0:
                    possible.append(col+'_num')
            except:
                pass
        numeric_candidates = possible
    
    numeric_field = numeric_candidates[0] if numeric_candidates else df.select_dtypes(include='number').columns[0]
    print(f"Using numeric field: {numeric_field}")
    
    # Try to find a group field
    group_candidates = [col for col in df.columns if col.lower().startswith('group') or col.lower().startswith('family') or col.lower().startswith('scop') or 'class' in col.lower() or 'superfam' in col.lower() or 'type' in col.lower()]
    group_field = group_candidates[0] if len(group_candidates) else df.columns[0]
    print(f"Grouping by field: {group_field}")

    # Filter
    threshold = df[numeric_field].astype(float).mean()
    filtered_df = df[df[numeric_field].astype(float) > threshold].copy()
    print(f"Filtered to rows with {numeric_field} > {threshold:.2f} (mean value)")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()) / filtered_df[numeric_field].astype(float).std()
    print(f"\nShowing normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        display(grouped_df.head())
else:
    print("No usable main DataFrame found for EDA.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field and the grouped means for subsets.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of numeric field
if df is not None and numeric_field in df.columns:
    plt.figure(figsize=(7, 4))
    df[numeric_field].astype(float).hist(bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.grid(True)
    plt.show()
    
    # Bar plot for grouped means (if available)
    if 'grouped_df' in locals() and grouped_df.shape[0] < 25:
        grouped_df.plot(kind='bar', figsize=(10,4), legend=False)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to:
- Programmatically access and explore a Croissant-structured dataset using its schema URL and `mlcroissant`.
- Enumerate all record sets, inspect their available field `@id`s, and flexibly extract tabular records by reference.
- Perform quick EDA on numeric features and visualize distributions in a reproducible, schema-driven workflow.

This pipeline can be adapted to any [ML Commons Croissant](https://mlcommons.org/working-groups/croissant/) structured dataset by referencing entities via their unique `@id` as shown above.

_For further analyses, you can apply machine learning, enrichment, or domain-specific logic, ensuring entity mapping and field manipulations always use the canonical `@id` values from the Croissant description._